# End-to-end differentiable micrograph simulation

Jointly recover an in-plane atomic **shift** and a small 3D **rotation** (axis-angle) of apoferritin (**6DRV**) by gradient descent through:

```text
pose → dry ESP → multislice → CTF → |ψ|² → expected counts
```

Poisson and continuum solvent stay out of the backward graph. The target is a noise-free micrograph at a known pose.

Related: [`simulate_micrograph_from_pdb.ipynb`](simulate_micrograph_from_pdb.ipynb).


In [ ]:
import mmdf
import torch
from matplotlib import pyplot as plt
from torch_calculate_electrostatic_potential import (
    GridConfig,
    default_sublattice_radius,
    potential_from_structure_3d,
)
from torch_scattering import multislice
from torch_structure_manipulation import (
    AtomicStructure,
    annotate_bonding_environments,
    apply_rotation,
    apply_rotation_to_coords,
    center_structure,
    create_rotation_matrix_from_euler,
)

from torch_simulate_image import (
    CtfConfig,
    FluenceConfig,
    MicrographSimulationConfig,
    PoissonConfig,
    simulate_micrograph,
)

## Parameters


In [ ]:
PIXEL_SIZE = 2.0  # Å
VOLTAGE_KV = 300.0
PADDING_A = 30.0  # room for shift + small rotation
DOSE_E_PER_A2 = 50.0
DEFOCUS_UM = 1.5
SHIFT_TRUE_YX_A = (6.0, -4.0)  # Å (y, x)
# Small true rotation as axis-angle (radians, xyz); |ω| ≈ 10°
AXIS_ANGLE_TRUE_XYZ = (0.12, -0.08, 0.10)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

## Helpers

Pose is axis-angle ω (xyz, radians) plus in-plane shift. Rotation about the origin (centered structure), then translation. Optics are fixed via `simulate_micrograph` (Poisson off).


In [ ]:
config = MicrographSimulationConfig(
    pixel_size=PIXEL_SIZE,
    ctf=CtfConfig(defocus_um=DEFOCUS_UM, voltage_kv=VOLTAGE_KV),
    fluence=FluenceConfig(dose_e_per_A2=DOSE_E_PER_A2),
    poisson=PoissonConfig(apply=False),
)


def axis_angle_to_rotation_matrix_xyz(v: torch.Tensor) -> torch.Tensor:
    """Rodrigues formula. v: (3,) axis-angle in xyz (radians)."""
    theta = v.norm(p=2).clamp(min=1e-7)
    k = v / theta
    zeros = torch.zeros((), dtype=v.dtype, device=v.device)
    K = torch.stack(
        [
            torch.stack([zeros, -k[2], k[1]]),
            torch.stack([k[2], zeros, -k[0]]),
            torch.stack([-k[1], k[0], zeros]),
        ]
    )
    identity = torch.eye(3, dtype=v.dtype, device=v.device)
    return identity + torch.sin(theta) * K + (1.0 - torch.cos(theta)) * (K @ K)


def apply_pose(
    structure: AtomicStructure,
    axis_angle_xyz: torch.Tensor,
    shift_yx: torch.Tensor,
) -> AtomicStructure:
    """Rotate about origin (xyz axis-angle), then translate in y/x (Å)."""
    R = axis_angle_to_rotation_matrix_xyz(axis_angle_xyz)
    rotated = apply_rotation_to_coords(
        structure.positions_zyx, R, center_point=None, zyx=True
    )
    zero_z = torch.zeros((), device=shift_yx.device, dtype=shift_yx.dtype)
    delta = torch.stack((zero_z, shift_yx[0], shift_yx[1]))
    return structure.with_positions(rotated + delta)


def simulate_dry_expected(
    structure: AtomicStructure,
    grid: GridConfig,
) -> torch.Tensor:
    """Positions → ESP → multislice → micrograph (expected counts)."""
    potential = potential_from_structure_3d(
        structure,
        grid,
        scattering_factors="peng_elemental",
    )
    wave = multislice(potential, pixel_size=PIXEL_SIZE, voltage=VOLTAGE_KV)
    return simulate_micrograph(wave, config)

## 1. Structure and grid

Dry apoferritin on a padded grid so the true pose stays inside the box.


In [ ]:
atoms = mmdf.read("pdb:6drv")
annotated = annotate_bonding_environments(atoms, include_hydrogens=False)
centered = center_structure(annotated, center_point=(0.0, 0.0, 0.0), zyx=False)
rotation = create_rotation_matrix_from_euler(
    torch.tensor([0.0, 30.0, 0.0]),
    order="ZYZ",
    degrees=True,
)
posed = apply_rotation(centered, rotation, zyx=False)
structure = AtomicStructure.from_dataframe(posed, device=DEVICE)
print(f"{structure.num_atoms} atoms on {structure.device}")

In [ ]:
positions = structure.positions_zyx
mins = positions.amin(dim=0) - PADDING_A
maxs = positions.amax(dim=0) + PADDING_A

grid = GridConfig.from_voxel_size_and_corner_points(
    voxel_size=(PIXEL_SIZE, PIXEL_SIZE, PIXEL_SIZE),
    left_bottom_point=tuple(mins.tolist()),
    right_upper_point=tuple(maxs.tolist()),
    sublattice_radius=default_sublattice_radius(PIXEL_SIZE),
    device=DEVICE,
)
print("grid shape (Z,Y,X):", tuple(int(n) for n in grid.grid_shape.tolist()))

## 2. Target at the true pose

Noise-free expected counts at the true axis-angle + shift. Optimization starts from identity / zero.


In [ ]:
true_axis_angle = torch.tensor(AXIS_ANGLE_TRUE_XYZ, device=DEVICE, dtype=torch.float32)
true_shift = torch.tensor(SHIFT_TRUE_YX_A, device=DEVICE, dtype=torch.float32)
structure_true = apply_pose(structure, true_axis_angle, true_shift)
target = simulate_dry_expected(structure_true, grid).detach()

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(target.cpu().numpy(), cmap="gray")
ax.set_title(
    f"Target\nshift={SHIFT_TRUE_YX_A} Å, |ω|={float(true_axis_angle.norm()):.3f} rad"
)
fig.colorbar(im, ax=ax, fraction=0.046)
plt.show()

## 3. Confirm gradients reach pose parameters

One backward pass through ESP → multislice → optics.


In [ ]:
axis_angle_check = torch.nn.Parameter(
    torch.zeros(3, device=DEVICE, dtype=torch.float32)
)
shift_check = torch.nn.Parameter(torch.zeros(2, device=DEVICE, dtype=torch.float32))
pred_check = simulate_dry_expected(
    apply_pose(structure, axis_angle_check, shift_check), grid
)
loss_check = torch.mean((pred_check - target) ** 2)
loss_check.backward()
assert axis_angle_check.grad is not None and shift_check.grad is not None
print(
    "grad ω:",
    axis_angle_check.grad.detach().cpu().tolist(),
    "|grad ω|=",
    float(axis_angle_check.grad.norm()),
)
print(
    "grad shift_yx:",
    shift_check.grad.detach().cpu().tolist(),
    "|grad shift|=",
    float(shift_check.grad.norm()),
)

## 4. Joint gradient descent

Adam on ω and shift together (separate learning rates). Each step rebuilds the full dry forward model.


In [ ]:
axis_angle = torch.nn.Parameter(torch.zeros(3, device=DEVICE, dtype=torch.float32))
shift = torch.nn.Parameter(torch.zeros(2, device=DEVICE, dtype=torch.float32))
optimizer = torch.optim.Adam(
    [
        {"params": [axis_angle], "lr": 0.02},
        {"params": [shift], "lr": 0.4},
    ]
)
n_steps = 50
loss_hist = []
shift_hist = []
angle_hist = []  # |ω| in degrees

for step in range(n_steps):
    optimizer.zero_grad(set_to_none=True)
    pred = simulate_dry_expected(apply_pose(structure, axis_angle, shift), grid)
    loss = torch.mean((pred - target) ** 2)
    loss.backward()
    optimizer.step()
    loss_hist.append(float(loss.detach()))
    shift_hist.append(shift.detach().cpu().tolist())
    angle_hist.append(float(axis_angle.detach().norm()) * 180.0 / 3.141592653589793)
    if step % 5 == 0 or step == n_steps - 1:
        sy, sx = shift_hist[-1]
        print(
            f"step {step:3d}  loss={loss_hist[-1]:.4f}  "
            f"shift_yx=({sy:.3f}, {sx:.3f}) Å  "
            f"|ω|={angle_hist[-1]:.2f}°  "
            f"ω={axis_angle.detach().cpu().tolist()}"
        )

sy, sx = shift.detach().cpu().tolist()
print(f"\ntrue shift={SHIFT_TRUE_YX_A}  recovered=({sy:.3f}, {sx:.3f}) Å")
print(
    f"true ω={AXIS_ANGLE_TRUE_XYZ}  "
    f"recovered={tuple(round(float(x), 4) for x in axis_angle.detach().cpu())}"
)
print(
    f"true |ω|={float(true_axis_angle.norm()) * 180 / 3.141592653589793:.2f}°  "
    f"recovered |ω|={float(axis_angle.detach().norm()) * 180 / 3.141592653589793:.2f}°"
)

In [ ]:
shift_arr = torch.tensor(shift_hist)
true_angle_deg = float(true_axis_angle.norm()) * 180.0 / 3.141592653589793

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(loss_hist)
axes[0].set_xlabel("step")
axes[0].set_ylabel("MSE")
axes[0].set_title("Pose GD loss")

axes[1].plot(shift_arr[:, 0], label="y")
axes[1].plot(shift_arr[:, 1], label="x")
axes[1].axhline(SHIFT_TRUE_YX_A[0], color="C0", ls="--", alpha=0.7)
axes[1].axhline(SHIFT_TRUE_YX_A[1], color="C1", ls="--", alpha=0.7)
axes[1].set_xlabel("step")
axes[1].set_ylabel("shift (Å)")
axes[1].set_title("Shift trajectory")
axes[1].legend()

axes[2].plot(angle_hist, label="|ω| estimate")
axes[2].axhline(true_angle_deg, color="C1", ls="--", label="true |ω|")
axes[2].set_xlabel("step")
axes[2].set_ylabel("|ω| (deg)")
axes[2].set_title("Rotation magnitude")
axes[2].legend()
plt.tight_layout()
plt.show()

## Takeaways

- Gradients flow through dry ESP → multislice → `simulate_micrograph` for both **translation** and **rotation** when positions are tensors.
- Axis-angle is a convenient local parameterization for small angular refine (same idea as `torch-fit-in-map`).
- Keep Poisson (and continuum solvent) out of the optimize loop; use them for observations / display.
